In [1]:
train_sets = ['/kaggle/input/datasets/michalbarnas/training-data/sample_23.csv'] # do not touch
test_set = '/kaggle/input/datasets/michalbarnas/training-data/sample_24.csv' # do not touch
max_rows = -1 # -1 for the whole file

# Approximate Gaussian Process (AGP) Configuration
# AGP models use inducing points and minibatches, so they scale better than exact GPs.
# Keep these conservative if you want faster notebook runs.
GP_NUM_INDUCING_POINTS = 1024  # Number of inducing points for the variational AGP
GP_EPOCHS = 10  # Training epochs for the variational objective
GP_MINIBATCH_SIZE = 512  # Minibatch size for AGP training
GP_LEARNING_RATE = 0.005  # Adam learning rate
GP_USE_LOG1P_TARGET = True  # Train in log1p-space and invert with expm1 at prediction time
EXPORT_COMBINED_RESULTS = False # Export only the combined forecasts when enabled

# Run toggles
RUN_ALL_DATA = False
RUN_GLOBAL_TRAIN_CLUSTER = False
RUN_GLOBAL_TRAIN_CLUSTER_COMBINED = False
RUN_CLUSTER_SPECIFIC_COMBINED = True
RUN_GLOBAL_WITH_CLUSTER_FEATURE = False

# Insert below paths to your clusterings (format required as a CSV: ID,shapelet_cluster)
clusterings = [
                  # 'outputs/kshape/kshape_7.csv',
                  # 'outputs/feature/case1_clusters.csv',
                  # 'outputs/feature/case2_clusters.csv',
                  # 'outputs/feature/case3_clusters.csv',
                  # 'outputs/feature/case4_clusters.csv',
                  '/kaggle/input/datasets/michalbarnas/cluster5/case5_clusters.csv',
                  '/kaggle/input/datasets/michalbarnas/kmedoid/kmedoid_cluster_labels_sample_23.csv',
            ]

In [2]:
%pip install gpytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.2/291.2 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.8/174.8 kB 14.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
import numpy as np
import pandas as pd
import warnings

import torch
import gpytorch
from torch.utils.data import DataLoader, TensorDataset

from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')

In [4]:
# Helper functions
def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["dow"] = out["date"].dt.dayofweek
    out["month"] = out["date"].dt.month
    out["day"] = out["date"].dt.day
    out["dayofyear"] = out["date"].dt.dayofyear

    out["doy_sin"] = np.sin(2 * np.pi * out["dayofyear"] / 366.0)
    out["doy_cos"] = np.cos(2 * np.pi * out["dayofyear"] / 366.0)
    out["dow_sin"] = np.sin(2 * np.pi * out["dow"] / 7.0)
    out["dow_cos"] = np.cos(2 * np.pi * out["dow"] / 7.0)
    return out


def add_lag_features(df: pd.DataFrame, group_col: str, target_col: str) -> pd.DataFrame:
    out = df.copy()
    g = out.groupby(group_col)[target_col]

    out["lag_1"] = g.shift(1)
    out["lag_7"] = g.shift(7)
    out["lag_14"] = g.shift(14)
    out["lag_28"] = g.shift(28)

    out["roll_mean_7"] = g.shift(1).rolling(7).mean().reset_index(level=0, drop=True)
    out["roll_mean_28"] = g.shift(1).rolling(28).mean().reset_index(level=0, drop=True)
    return out


def build_cluster_mapping(clustering_df):
    clustering_df = clustering_df.copy()
    clustering_df.columns = clustering_df.columns.str.strip()

    id_candidates = [c for c in clustering_df.columns if c.lower() == "id"]
    cluster_candidates = [c for c in clustering_df.columns if "cluster" in c.lower()]

    cluster_id_col = id_candidates[0] if id_candidates else clustering_df.columns[0]
    cluster_col = cluster_candidates[0] if cluster_candidates else clustering_df.columns[1]

    cluster_map = clustering_df[[cluster_id_col, cluster_col]].dropna().drop_duplicates().copy()
    cluster_map[cluster_id_col] = cluster_map[cluster_id_col].astype(str)

    cluster_values = sorted(pd.unique(cluster_map[cluster_col]))
    cluster_value_to_code = {v: i for i, v in enumerate(cluster_values)}

    id_to_cluster_value = dict(zip(cluster_map[cluster_id_col], cluster_map[cluster_col]))
    id_to_cluster_code = {
        item_id: int(cluster_value_to_code[value])
        for item_id, value in id_to_cluster_value.items()
    }

    return cluster_map, cluster_id_col, cluster_col, id_to_cluster_value, id_to_cluster_code


def train_model_with_cluster_feature(train_wide_in, id_col, id_to_cluster_code):
    def add_cluster_code_feature(df):
        out = df.copy()
        out["cluster_code"] = out[id_col].astype(str).map(id_to_cluster_code).fillna(-1).astype(float)
        return out

    train_model_fn = globals()["train_model"]
    return train_model_fn(
        train_wide_in,
        id_col,
        extra_feature_builder=add_cluster_code_feature,
        extra_feature_cols=["cluster_code"],
    )


class AGPModel(gpytorch.models.ApproximateGP):
    def __init__(self, inducing_points, input_dim):
        variational_distribution = gpytorch.variational.CholeskyVariationalDistribution(inducing_points.size(0))
        variational_strategy = gpytorch.variational.VariationalStrategy(
            self,
            inducing_points,
            variational_distribution,
            learn_inducing_locations=True,
        )
        super().__init__(variational_strategy)
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.AdditiveKernel(
                gpytorch.kernels.RBFKernel(ard_num_dims=input_dim),
                gpytorch.kernels.MaternKernel(nu=2.5, ard_num_dims=input_dim),
            )
        )

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)


class AGPBundle:
    def __init__(self, gp_model, likelihood, feature_scaler, target_scaler, device, feature_cols, use_log1p_target=False):
        self.gp_model = gp_model
        self.likelihood = likelihood
        self.scaler = feature_scaler
        self.target_scaler = target_scaler
        self.device = device
        self.feature_cols = feature_cols
        self.use_log1p_target = bool(use_log1p_target)

In [5]:
def preprocess_test_data(test_wide, id_col):
    test_cols = pd.Index(test_wide.columns)
    test_parsed = pd.to_datetime(test_cols, format="%Y-%m-%d", errors="coerce")
    test_date_mask = (test_cols != id_col) & test_parsed.notna()
    test_date_cols = test_cols[test_date_mask].tolist()
    future_dates = np.array(sorted(pd.to_datetime(test_date_cols)))

    test_long_actual = test_wide.melt(id_vars=id_col, value_vars=test_date_cols, var_name="date", value_name="y_true")
    test_long_actual["date"] = pd.to_datetime(test_long_actual["date"])
    return future_dates, test_long_actual

In [6]:
def train_model(train_wide, id_col, extra_feature_builder=None, extra_feature_cols=None):
    train_cols = pd.Index(train_wide.columns)
    train_parsed = pd.to_datetime(train_cols, format="%Y-%m-%d", errors="coerce")
    train_date_mask = (train_cols != id_col) & train_parsed.notna()
    train_date_cols = train_cols[train_date_mask].tolist()

    train_long = train_wide.melt(id_vars=id_col, value_vars=train_date_cols, var_name="date", value_name="y")
    train_long["date"] = pd.to_datetime(train_long["date"])
    train_long = train_long.sort_values([id_col, "date"]).reset_index(drop=True)

    print("Adding features")
    train_feat = add_calendar_features(train_long)
    train_feat = add_lag_features(train_feat, group_col=id_col, target_col="y")

    if extra_feature_builder is not None:
        train_feat = extra_feature_builder(train_feat)

    feature_cols = [
        "dow", "month", "day", "dayofyear",
        "doy_sin", "doy_cos", "dow_sin", "dow_cos",
        "lag_1", "lag_7", "lag_14", "lag_28",
        "roll_mean_7", "roll_mean_28",
    ]
    if extra_feature_cols:
        feature_cols.extend(extra_feature_cols)

    train_model_df = train_feat.dropna(subset=feature_cols + ["y"]).copy()
    if len(train_model_df) == 0:
        raise ValueError("No rows available after feature engineering.")

    # Use all available training data without downsampling
    print(f"Training with {len(train_model_df)} samples")

    X_train = train_model_df[feature_cols].to_numpy(dtype=float)
    y_train_raw = train_model_df["y"].to_numpy(dtype=float)

    use_log1p_target = bool(globals().get("GP_USE_LOG1P_TARGET", False))
    if use_log1p_target:
        y_train_raw = np.clip(y_train_raw, a_min=0.0, a_max=None)
        y_train_for_model = np.log1p(y_train_raw)
    else:
        y_train_for_model = y_train_raw

    print("Standardizing features and target")
    feature_scaler = StandardScaler()
    X_train_scaled = feature_scaler.fit_transform(X_train)
    target_scaler = StandardScaler()
    y_train_scaled = target_scaler.fit_transform(y_train_for_model.reshape(-1, 1)).ravel()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    x_tensor = torch.from_numpy(X_train_scaled).float()
    y_tensor = torch.from_numpy(y_train_scaled).float()

    num_inducing = min(int(globals().get("GP_NUM_INDUCING_POINTS", 256)), len(X_train_scaled))
    rng = np.random.default_rng(42)
    print(f"Initializing {num_inducing} inducing points using k-means centers")

    try:
        if num_inducing < len(X_train_scaled):
            kmeans_batch_size = min(max(1024, num_inducing * 4), len(X_train_scaled))
            kmeans = MiniBatchKMeans(
                n_clusters=num_inducing,
                random_state=42,
                n_init=10,
                batch_size=kmeans_batch_size,
            )
            kmeans.fit(X_train_scaled)
            inducing_np = kmeans.cluster_centers_.astype(np.float32, copy=False)
        else:
            inducing_np = X_train_scaled.astype(np.float32, copy=False)
    except Exception as exc:
        # Keep training robust if k-means fails for any edge case.
        print(f"K-means inducing init failed ({exc}); falling back to random subset.")
        inducing_idx = rng.choice(len(X_train_scaled), size=num_inducing, replace=False)
        inducing_np = X_train_scaled[inducing_idx].astype(np.float32, copy=False)

    inducing_points = torch.from_numpy(inducing_np).float().to(device)

    train_dataset = TensorDataset(x_tensor, y_tensor)
    batch_size = min(int(globals().get("GP_MINIBATCH_SIZE", 2048)), len(train_dataset))
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=False)

    print("Fitting AGP model")
    gp_model = AGPModel(inducing_points=inducing_points, input_dim=X_train_scaled.shape[1]).to(device)
    likelihood = gpytorch.likelihoods.GaussianLikelihood().to(device)
    gp_model.train()
    likelihood.train()

    optimizer = torch.optim.Adam(
        list(gp_model.parameters()) + list(likelihood.parameters()),
        lr=float(globals().get("GP_LEARNING_RATE", 0.01)),
    )
    mll = gpytorch.mlls.VariationalELBO(likelihood, gp_model, num_data=len(train_dataset))

    num_epochs = int(globals().get("GP_EPOCHS", 4))
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            output = gp_model(batch_x)
            loss = -mll(output, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += float(loss.item()) * len(batch_x)

        avg_loss = epoch_loss / max(len(train_dataset), 1)
        print(f"Epoch {epoch + 1}/{num_epochs} - loss: {avg_loss:.4f}")

    gp_model.eval()
    likelihood.eval()

    model = AGPBundle(
        gp_model=gp_model,
        likelihood=likelihood,
        feature_scaler=feature_scaler,
        target_scaler=target_scaler,
        device=device,
        feature_cols=feature_cols,
        use_log1p_target=use_log1p_target,
    )

    return model, feature_cols


In [7]:
def predict(model, future_dates, id_values, train_wide, id_col, feature_cols, extra_feature_builder=None):
    n_ids = len(id_values)
    n_steps = len(future_dates)
    pred_matrix = np.empty((n_ids, n_steps), dtype=float)
    uncertainty_matrix = np.empty((n_ids, n_steps), dtype=float)  # AGP provides uncertainty estimates

    train_date_cols = [
        c for c in train_wide.columns
        if c != id_col and pd.notna(pd.to_datetime(c, format="%Y-%m-%d", errors="coerce"))
    ]

    if len(train_date_cols) == 0:
        raise ValueError("No valid date columns found in train data.")

    train_wide_indexed = train_wide.set_index(id_col)

    history_by_id = {}
    for item_id in id_values:
        if item_id in train_wide_indexed.index:
            hist_series = train_wide_indexed.loc[item_id, train_date_cols]
            history_by_id[item_id] = np.asarray(hist_series, dtype=float).tolist()
        else:
            history_by_id[item_id] = []

    print("Predicting with AGP...")
    scaler = model.scaler
    target_scaler = model.target_scaler
    gp_model = model.gp_model
    likelihood = model.likelihood
    device = model.device
    use_log1p_target = bool(getattr(model, "use_log1p_target", False))

    gp_model.eval()
    likelihood.eval()

    for step_idx, dt in enumerate(future_dates):
        dow = dt.dayofweek
        month = dt.month
        day = dt.day
        dayofyear = dt.dayofyear

        doy_sin = np.sin(2 * np.pi * dayofyear / 366.0)
        doy_cos = np.cos(2 * np.pi * dayofyear / 366.0)
        dow_sin = np.sin(2 * np.pi * dow / 7.0)
        dow_cos = np.cos(2 * np.pi * dow / 7.0)

        X_batch = np.empty((n_ids, len(feature_cols)), dtype=float)

        for i, item_id in enumerate(id_values):
            hist = history_by_id[item_id]

            lag_1 = hist[-1] if len(hist) >= 1 else np.nan
            lag_7 = hist[-7] if len(hist) >= 7 else lag_1
            lag_14 = hist[-14] if len(hist) >= 14 else lag_1
            lag_28 = hist[-28] if len(hist) >= 28 else lag_1

            roll_mean_7 = np.mean(hist[-7:]) if len(hist) >= 7 else lag_1
            roll_mean_28 = np.mean(hist[-28:]) if len(hist) >= 28 else lag_1

            row_features = [
                dow, month, day, dayofyear,
                doy_sin, doy_cos, dow_sin, dow_cos,
                lag_1, lag_7, lag_14, lag_28,
                roll_mean_7, roll_mean_28,
            ]

            if extra_feature_builder is not None:
                extra_features = extra_feature_builder(item_id=item_id, dt=dt, hist=hist)
                row_features.extend(extra_features)

            X_batch[i, :] = row_features

        # Scale features before prediction
        X_batch_scaled = scaler.transform(X_batch)
        x_tensor = torch.from_numpy(X_batch_scaled).float().to(device)

        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            pred_dist = likelihood(gp_model(x_tensor))
            mean_scaled = pred_dist.mean.detach().cpu().numpy()
            std_scaled = pred_dist.variance.clamp_min(1e-12).sqrt().detach().cpu().numpy()

        y_pred_model_space = mean_scaled * target_scaler.scale_[0] + target_scaler.mean_[0]
        y_std_model_space = std_scaled * target_scaler.scale_[0]

        if use_log1p_target:
            # Inverse of log1p target transform with delta-method std approximation.
            y_pred_batch = np.expm1(y_pred_model_space)
            y_pred_batch = np.clip(y_pred_batch, a_min=0.0, a_max=None)
            y_std_batch = np.exp(y_pred_model_space) * y_std_model_space
        else:
            y_pred_batch = y_pred_model_space
            y_std_batch = y_std_model_space

        pred_matrix[:, step_idx] = y_pred_batch
        uncertainty_matrix[:, step_idx] = y_std_batch

        for i, item_id in enumerate(id_values):
            history_by_id[item_id].append(float(y_pred_batch[i]))

    pred_long = pd.DataFrame(
        {
            id_col: np.repeat(id_values, n_steps),
            "date": np.tile(future_dates, n_ids),
            "y_pred": pred_matrix.reshape(-1),
            "y_std": uncertainty_matrix.reshape(-1),  # Include uncertainty estimates
        }
    )

    return pred_long


def predict_with_cluster_feature(model, future_dates, id_values, train_wide_in, id_col, feature_cols, id_to_cluster_code):
    def add_cluster_feature(item_id, dt, hist):
        _ = (dt, hist)
        return [float(id_to_cluster_code.get(item_id, -1))]

    return predict(
        model,
        future_dates,
        id_values,
        train_wide_in,
        id_col,
        feature_cols,
        extra_feature_builder=add_cluster_feature,
    )

In [8]:
def evaluate(pred_long, train_long, test_long_actual, id_col):
    train_rows = len(train_long)
    if train_rows == 0:
        print("No training rows provided.")

    eval_df = pred_long.merge(test_long_actual, on=[id_col, "date"], how="left")

    valid_mask = eval_df["y_true"].notna()

    metrics = {
        "mae": np.nan,
        "rmse": np.nan,
        "nrmse_mean": np.nan,
        "cvrmse_pct": np.nan,
        "nrmse_range": np.nan,
        "n_eval_points": int(valid_mask.sum()),
    }

    if valid_mask.any():
        y_true = eval_df.loc[valid_mask, "y_true"]
        y_pred = eval_df.loc[valid_mask, "y_pred"]

        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))

        mean_true = float(y_true.mean())
        y_true_range = float(y_true.max() - y_true.min())

        nrmse_mean = (rmse / mean_true) if abs(mean_true) > 1e-12 else np.nan
        cvrmse_pct = 100.0 * nrmse_mean if np.isfinite(nrmse_mean) else np.nan
        nrmse_range = (rmse / y_true_range) if y_true_range > 1e-12 else np.nan

        metrics.update(
            {
                "mae": float(mae),
                "rmse": float(rmse),
                "nrmse_mean": float(nrmse_mean) if np.isfinite(nrmse_mean) else np.nan,
                "cvrmse_pct": float(cvrmse_pct) if np.isfinite(cvrmse_pct) else np.nan,
                "nrmse_range": float(nrmse_range) if np.isfinite(nrmse_range) else np.nan,
            }
        )

    pred_wide = pred_long.copy()
    pred_wide["date"] = pred_wide["date"].dt.strftime("%Y-%m-%d")
    
    # Keep y_std if available (from AGP predictions)
    if "y_std" in pred_wide.columns:
        pred_wide = pred_wide.pivot_table(index=id_col, columns="date", values="y_pred").reset_index()
    else:
        pred_wide = pred_wide.pivot(index=id_col, columns="date", values="y_pred").reset_index()

    metrics["n_pred_rows"] = int(len(pred_long))

    return metrics, pred_wide, eval_df


def print_evaluation_summary(results):
    if len(results) == 0:
        print("No evaluation results to summarize.")
        return

    summary_df = pd.DataFrame(results)

    cols = [
        "clustering_used",
        "label",
        "n_eval_points",
        "mae",
        "rmse",
        "cvrmse_pct",
        "nrmse_range",
    ]
    cols = [c for c in cols if c in summary_df.columns]

    print("Evaluation Summary")
    print(summary_df[cols].to_string(index=False))

    numeric_cols = ["mae", "rmse", "cvrmse_pct", "nrmse_range"]
    available_numeric = [c for c in numeric_cols if c in summary_df.columns]

    if len(available_numeric) > 0:
        print("\nAverages")
        print(summary_df[available_numeric].mean(numeric_only=True).to_string())

In [9]:
def run(train_data, test_data, id_col, label="run"):
    train_data = train_data.copy()
    test_data = test_data.copy()

    # Normalize incoming column names and requested id column to avoid whitespace mismatches.
    train_data.columns = train_data.columns.str.strip()
    test_data.columns = test_data.columns.str.strip()
    id_col = id_col.strip()

    if id_col not in train_data.columns:
        raise ValueError(f"ID column '{id_col}' not found in train data. Available: {list(train_data.columns[:5])}...")
    if id_col not in test_data.columns:
        raise ValueError(f"ID column '{id_col}' not found in test data. Available: {list(test_data.columns[:5])}...")

    model, feature_cols = train_model(train_data, id_col)

    id_values = np.array(sorted(pd.unique(test_data[id_col].dropna())))
    if len(id_values) == 0:
        raise ValueError("No IDs found in test data.")

    future_dates, test_long_actual = preprocess_test_data(test_data, id_col)

    pred_long = predict(model, future_dates, id_values, train_data, id_col, feature_cols)

    metrics, pred_wide, eval_df = evaluate(
        pred_long,
        train_data.melt(id_vars=id_col, var_name="date", value_name="y"),
        test_long_actual,
        id_col,
    )
    metrics["label"] = label

    return metrics, pred_wide, eval_df

In [10]:
# Data loading
train_path = train_sets[0] if isinstance(train_sets, (list, tuple)) else train_sets
test_path = test_set

if max_rows != -1:
    train_wide = pd.read_csv(train_path, nrows=max_rows)
    test_wide = pd.read_csv(test_path, nrows=max_rows)
else:
    train_wide = pd.read_csv(train_path)
    test_wide = pd.read_csv(test_path)

results = []

run_all_data = bool(globals().get("RUN_ALL_DATA", True))
run_global_train_cluster = bool(globals().get("RUN_GLOBAL_TRAIN_CLUSTER", True))
run_global_train_cluster_combined = bool(globals().get("RUN_GLOBAL_TRAIN_CLUSTER_COMBINED", True))
run_cluster_specific_combined = bool(globals().get("RUN_CLUSTER_SPECIFIC_COMBINED", True))
run_global_with_cluster_feature = bool(globals().get("RUN_GLOBAL_WITH_CLUSTER_FEATURE", True))
export_combined_results = bool(globals().get("EXPORT_COMBINED_RESULTS", False))

if run_all_data:
    print("Running without clusters")
    base_metrics, _, _ = run(
        train_wide,
        test_wide,
        id_col="ID",
        label="all_data",
    )
    base_metrics["clustering_used"] = "none"
    results.append(base_metrics)
else:
    print("Skipping all-data forecast (RUN_ALL_DATA=False)")

train_norm = train_wide.copy()
train_norm.columns = train_norm.columns.str.strip()
train_norm["ID"] = train_norm["ID"].astype(str)

test_norm = test_wide.copy()
test_norm.columns = test_norm.columns.str.strip()
test_norm["ID"] = test_norm["ID"].astype(str)


def build_train_long(train_wide_in, id_col):
    train_cols = pd.Index(train_wide_in.columns)
    train_date_mask = (train_cols != id_col) & pd.to_datetime(
        train_cols, format="%Y-%m-%d", errors="coerce"
    ).notna()
    train_date_cols = train_cols[train_date_mask].tolist()
    train_long = train_wide_in.melt(
        id_vars=id_col, value_vars=train_date_cols, var_name="date", value_name="y"
    )
    train_long["date"] = pd.to_datetime(train_long["date"])
    return train_long


def evaluate_and_record(pred_long, train_wide_in, test_wide_in, id_col, label, clustering_used):
    _, test_long_actual = preprocess_test_data(test_wide_in, id_col)
    train_long = build_train_long(train_wide_in, id_col)

    metrics, _, _ = evaluate(
        pred_long,
        train_long,
        test_long_actual,
        id_col,
    )
    metrics["label"] = label
    metrics["clustering_used"] = clustering_used
    results.append(metrics)


def export_combined_pred_wide(combined_pred_wide, clustering_used, export_label):
    if not export_combined_results:
        return

    export_name = f"../data/processed/sklearn_forecast_2024_{export_label}_{clustering_used}.csv"
    combined_pred_wide.to_csv(export_name, index=False)
    print(f"Exported combined results to {export_name}")


def evaluate_cluster_specific_combined(train_norm_in, test_norm_in, cluster_map, cluster_id_col, cluster_col, clustering_used):
    cluster_pred_wide_parts = []

    for _, group in cluster_map.groupby(cluster_col):
        cluster_value = group[cluster_col].iloc[0]
        cluster_ids = set(group[cluster_id_col].tolist())

        train_cluster = train_norm_in[train_norm_in["ID"].isin(cluster_ids)].copy()
        test_cluster = test_norm_in[test_norm_in["ID"].isin(cluster_ids)].copy()

        if len(train_cluster) == 0 or len(test_cluster) == 0:
            continue

        cluster_metrics, cluster_pred_wide, _ = run(
            train_cluster,
            test_cluster,
            id_col="ID",
            label=f"cluster_specific_train_cluster_{cluster_value}",
        )
        cluster_metrics["clustering_used"] = clustering_used
        results.append(cluster_metrics)
        cluster_pred_wide_parts.append(cluster_pred_wide)

    if len(cluster_pred_wide_parts) == 0:
        return

    combined_pred_wide = pd.concat(cluster_pred_wide_parts, ignore_index=True)
    combined_pred_wide = combined_pred_wide.drop_duplicates(subset=["ID"], keep="last")
    combined_pred_long = combined_pred_wide.melt(
        id_vars="ID", var_name="date", value_name="y_pred"
    )
    combined_pred_long["date"] = pd.to_datetime(combined_pred_long["date"])

    evaluate_and_record(
        combined_pred_long,
        train_norm_in,
        test_norm_in,
        "ID",
        "cluster_specific_train_combined",
        clustering_used,
    )
    export_combined_pred_wide(combined_pred_wide, clustering_used, "cluster_specific_combined")


def evaluate_global_with_cluster_feature(train_norm_in, test_norm_in, id_to_cluster_code, label, clustering_used):
    model_cf, feature_cols_cf = train_model_with_cluster_feature(train_norm_in, "ID", id_to_cluster_code)

    id_values = np.array(sorted(pd.unique(test_norm_in["ID"].dropna())))
    future_dates, _ = preprocess_test_data(test_norm_in, "ID")

    pred_long_cf = predict_with_cluster_feature(
        model_cf,
        future_dates,
        id_values,
        train_norm_in,
        "ID",
        feature_cols_cf,
        id_to_cluster_code,
    )

    evaluate_and_record(
        pred_long_cf,
        train_norm_in,
        test_norm_in,
        "ID",
        label,
        clustering_used,
    )


for clustering_path in clusterings:
    print("## CLUSTERING FILE:", clustering_path)
    clustering_used = clustering_path.split('/')[-1].replace('.csv', '')
    clustering_df = pd.read_csv(clustering_path)

    (
        cluster_map,
        cluster_id_col,
        cluster_col,
        id_to_cluster_value,
        id_to_cluster_code,
    ) = build_cluster_mapping(clustering_df)

    cluster_pred_wide_parts = []

    if run_global_train_cluster:
        for _, group in cluster_map.groupby(cluster_col):
            cluster_value = group[cluster_col].iloc[0]
            cluster_ids = set(group[cluster_id_col].tolist())
            test_cluster = test_norm[test_norm["ID"].isin(cluster_ids)].copy()
            if len(test_cluster) == 0:
                continue

            cluster_metrics, cluster_pred_wide, _ = run(
                train_norm,
                test_cluster,
                id_col="ID",
                label=f"global_train_cluster_{cluster_value}",
            )
            cluster_metrics["clustering_used"] = clustering_used
            results.append(cluster_metrics)
            cluster_pred_wide_parts.append(cluster_pred_wide)
    else:
        print(f"Skipping global per-cluster forecasts for {clustering_used} (RUN_GLOBAL_TRAIN_CLUSTER=False)")

    if run_global_train_cluster_combined and len(cluster_pred_wide_parts) > 0:
        combined_pred_wide = pd.concat(cluster_pred_wide_parts, ignore_index=True)
        combined_pred_wide = combined_pred_wide.drop_duplicates(subset=["ID"], keep="last")
        combined_pred_long = combined_pred_wide.melt(
            id_vars="ID", var_name="date", value_name="y_pred"
        )
        combined_pred_long["date"] = pd.to_datetime(combined_pred_long["date"])

        evaluate_and_record(
            combined_pred_long,
            train_norm,
            test_norm,
            "ID",
            "global_train_cluster_combined",
            clustering_used,
        )
        export_combined_pred_wide(combined_pred_wide, clustering_used, "global_train_combined")
    elif run_global_train_cluster_combined:
        print(f"Skipping combined global per-cluster forecast for {clustering_used} (no cluster predictions)")
    else:
        print(f"Skipping combined global per-cluster forecasts for {clustering_used} (RUN_GLOBAL_TRAIN_CLUSTER_COMBINED=False)")

    if run_cluster_specific_combined:
        evaluate_cluster_specific_combined(
            train_norm,
            test_norm,
            cluster_map,
            cluster_id_col,
            cluster_col,
            clustering_used=clustering_used,
        )
    else:
        print(f"Skipping cluster-specific combined forecasts for {clustering_used} (RUN_CLUSTER_SPECIFIC_COMBINED=False)")

    if run_global_with_cluster_feature:
        evaluate_global_with_cluster_feature(
            train_norm,
            test_norm,
            id_to_cluster_code,
            label="global_with_cluster_feature",
            clustering_used=clustering_used,
        )
    else:
        print(f"Skipping global-with-cluster-feature forecasts for {clustering_used} (RUN_GLOBAL_WITH_CLUSTER_FEATURE=False)")

print_evaluation_summary(results)


Skipping all-data forecast (RUN_ALL_DATA=False)
## CLUSTERING FILE: /kaggle/input/datasets/michalbarnas/cluster5/case5_clusters.csv
Skipping global per-cluster forecasts for case5_clusters (RUN_GLOBAL_TRAIN_CLUSTER=False)
Skipping combined global per-cluster forecasts for case5_clusters (RUN_GLOBAL_TRAIN_CLUSTER_COMBINED=False)
Adding features
Training with 51898 samples
Standardizing features and target
Initializing 1024 inducing points using k-means centers
Fitting AGP model
Epoch 1/10 - loss: 0.9924
Epoch 2/10 - loss: 0.5308
Epoch 3/10 - loss: 0.2603
Epoch 4/10 - loss: -0.0111
Epoch 5/10 - loss: -0.2851
Epoch 6/10 - loss: -0.5588
Epoch 7/10 - loss: -0.8289
Epoch 8/10 - loss: -1.0976
Epoch 9/10 - loss: -1.3586
Epoch 10/10 - loss: -1.6117
Predicting with AGP...
Adding features
Training with 952699 samples
Standardizing features and target
Initializing 1024 inducing points using k-means centers
Fitting AGP model
Epoch 1/10 - loss: 0.5788
Epoch 2/10 - loss: 0.3115
Epoch 3/10 - loss: 0.2